[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-external-memory.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239440-lesson-6-chatbot-w-summarizing-messages-and-external-memory)

# 带消息摘要和外部数据库记忆的聊天机器人

## 回顾

我们已经介绍了如何自定义图状态 Schema 和 reducer。

我们还展示了一些在图状态中裁剪或过滤消息的技巧。

我们已经在具有记忆功能的聊天机器人中使用了这些概念，该机器人能生成对话的运行摘要。

## 目标

但是，如果我们希望聊天机器人拥有可以无限期持久化的记忆呢？

现在，我们将介绍一些更高级的支持外部数据库的检查点器。

这里，我们将展示如何使用 [Sqlite 作为检查点器](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries)，但其他检查点器，如 Postgres 也是可用的！

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph-checkpoint-sqlite langchain_core langgraph langchain_deepseek

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

## Sqlite

这里一个好的起点是 [SqliteSaver 检查点器](https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries)。

Sqlite 是一个[小巧、快速、非常流行的](https://x.com/karpathy/status/1819490455664685297) SQL 数据库。

如果我们提供 `":memory:"`，它会创建一个内存中的 Sqlite 数据库。

In [ ]:
import sqlite3
# 内存中
conn = sqlite3.connect(":memory:", check_same_thread = False)

但是，如果我们提供一个数据库路径，那么它会为我们创建一个数据库！

In [ ]:
# 如果文件不存在则拉取，并连接到本地数据库
!mkdir -p state_db && [ ! -f state_db/example.db ] && wget -P state_db https://github.com/langchain-ai/langchain-academy/raw/main/module-2/state_db/example.db

db_path = "state_db/example.db"
conn = sqlite3.connect(db_path, check_same_thread=False)

In [ ]:
# 这是我们的检查点器
from langgraph.checkpoint.sqlite import SqliteSaver
memory = SqliteSaver(conn)

让我们重新定义我们的聊天机器人。

In [ ]:
from typing_extensions import Literal
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

from langgraph.graph import END
from langgraph.graph import MessagesState


model = ChatDeepSeek(model="deepseek-chat",temperature=0)

class State(MessagesState):
    summary: str

# 定义调用模型的逻辑
def call_model(state: State):
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，则将其添加进去
    if summary:
        
        # 将摘要添加到系统消息中
        system_message = f"Summary of conversation earlier: {summary}"

        # 将摘要追加到较新的消息之前
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

def summarize_conversation(state: State):
    
    # 首先，获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的摘要 prompt
    if summary:
        
        # 已经存在一个摘要
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
        
    else:
        summary_message = "Create a summary of the conversation above:"

    # 将 prompt 添加到对话历史中
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近 2 条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

# 判断是结束还是进行摘要
def should_continue(state: State)-> Literal ["summarize_conversation",END]:
    
    """返回要执行的下一个节点。"""
    
    messages = state["messages"]
    
    # 如果消息超过六条，则对对话进行摘要
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则可以直接结束
    return END

现在，我们只需用 sqlite 检查点器重新编译。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START

# 定义一个新图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为 conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

现在，我们可以多次调用图。

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
input_message = HumanMessage(content="hi! I'm Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="what's my name?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="i like the 49ers!")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

让我们确认状态已保存在本地。

In [ ]:
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

### 持久化状态

使用像 Sqlite 这样的数据库意味着状态是被持久化的！

例如，我们可以重新启动 notebook 内核，然后仍然可以从磁盘上的 Sqlite 数据库加载。

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}
graph_state = graph.get_state(config)
graph_state

## Studio

**⚠️ 注意**

自录制这些视频以来，我们已经更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而不是使用视频中展示的桌面应用。它现在被称为 _LangSmith Studio_，而不是 _LangGraph Studio_。详细的设置说明可在课程开头的"环境搭建"指南中找到。你可以在[这里](https://docs.langchain.com/langsmith/studio)找到 Studio 的描述，以及在[这里](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。
要在本地启动开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。
在 Studio 中加载 `chatbot`，它使用在 `module-2/studio/langgraph.json` 中设置的 `module-2/studio/chatbot.py`。